## Import packages

In [ ]:
import os, sys
import pandas as pd
# import pyBigWig
import cooler
import warnings
warnings.filterwarnings("ignore")
from loguru import logger

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/BICAN_pseudobulk"
outdir=None,
mode='raw'
res=100000
genome="mm10"
chrom_size_path="~/Ref/mm10/mm10_ucsc.nochrM.sizes"
metadata_path="~/Projects/mouse_pfc/clustering/5kb/cell_metadata_with_annotation.csv"
group_col='Subclass'
parent_col="Class"
sort_cols=None
stratify_col=None
stratify_figdir=None
dchic_env="dchic"
excluded_groups=['Low quality cells','doublet','Unk']
prepare_only=False

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/compartment"
outname = "3.Class.run_diffcomp.ipynb"


In [ ]:
workdir = os.path.expanduser(workdir)
if outdir is None:
    outdir=os.path.join(workdir,"compartment")
if sort_cols is None:
    sort_cols=parent_col
outdir=os.path.expanduser(outdir)
assert os.path.exists(outdir)
Rscript = os.path.join(os.path.abspath(os.path.join(os.path.dirname(sys.executable), '../../')),
                      f"{dchic_env}/bin/Rscript")
diff_comp_dir = os.path.join(outdir, f"{group_col}DiffComp")
# compartment_score_dir=os.path.join(outdir, f"{group_col}CompScore")
for d in [diff_comp_dir]:
    if not os.path.exists(d):
        os.mkdir(d)
chrom_size_path = os.path.expanduser(chrom_size_path)
metadata_path = os.path.expanduser(metadata_path)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes[chrom_sizes.index != 'chrY']
df_chr = chrom_sizes.to_frame()
df_chr['N'] = df_chr.index.to_series().apply(
	lambda x: int(x.lstrip('chr')) if x.lstrip('chr') not in ['X', 'Y'] else 30)  # sort by chrom int (1,2,3..)
df_chr.sort_values('N', inplace=True)
chrom_sizes = chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
	metadata = pd.read_csv(metadata_path, index_col=0)
else:
	metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
print(metadata.shape)
metadata.head()

In [ ]:
comp = pd.read_csv(os.path.join(workdir, "compartment",
                                f"{mode}_{group_col}", "compartment_score_pcs.tsv.gz"),
                   sep='\t', index_col=0)
print(comp.shape)
comp.head()

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not sort_cols is None:
    metadata.sort_values(sort_cols,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)

group_order=comp.columns.tolist()
if not parent_col is None:
	parent2group=metadata.loc[:,[parent_col,group_col]].drop_duplicates().groupby(parent_col)[group_col].apply(lambda x:x.unique().tolist()).to_dict()
else:
    parent2group=dict()
if not stratify_col is None:
    if stratify_figdir is None:
        metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str) #apply(lambda x:x.replace(' ','_')).
    else:
        metadata['GroupID']=metadata[group_col]
    stratify2group=metadata.loc[:,[stratify_col,'GroupID']].drop_duplicates().groupby(stratify_col)['GroupID'].apply(lambda x:x.unique().tolist()).to_dict()
    group2stratify=metadata.loc[:,[stratify_col,'GroupID']].drop_duplicates().set_index('GroupID')[stratify_col].to_dict()
    if not parent_col is None:
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    else:
        group2parent={} # group is either cell types (such ODC) or GroupID (such as ODC+Pu), parent is Class (such as Glut, Nonneuron)
else:
    stratify2group={}
    group2stratify={}
    if not parent_col is None:
        group2parent=metadata.loc[:,[parent_col,group_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        group2parent={}
group2parent['merged']='merged'
parent2group['merged']=metadata[group_col].unique().tolist()
print(group_order)
print(stratify2group)
print(group2stratify)
print(group2parent)
print(parent2group)

In [ ]:
binall = pd.DataFrame(index=comp.index)
binall['chrom'] = binall.index.str.split('-').str[0]
binall['start'] = binall.index.str.split('-').str[1].astype(int) * res
binall['end'] = binall['start'] + res

In [ ]:
cmds = []
use_dict=stratify2group if not stratify_col is None else parent2group
for group in use_dict:
	group_name=group.replace(' ','_')
	logger.info(group)
	group_dir = os.path.join(diff_comp_dir, group_name)
	common_cts=list(set(use_dict[group]) & set(group_order))
	if len(common_cts)==0:
		continue
	if not os.path.exists(group_dir):
		os.mkdir(group_dir)
	# if len(use_dict[group])<2: #skip one group case (would get error, cause there is no comparison)
	# 	continue
	for ct in use_dict[group]:
		if ct not in group_order:
			continue
		ct_name = ct.replace(' ','_')
		# dchic would throw error if there is special character in the name
		ct_dir = os.path.join(group_dir, f"{ct_name.replace('-', '_AND_').replace('+','_PLUS_')}_100Kb_pca/intra_pca/{ct_name.replace('-', '_AND_').replace('+','_PLUS_')}_100Kb_mat")
		if not os.path.exists(ct_dir):
			os.makedirs(ct_dir, exist_ok=True)
		tmp = binall.copy()
		tmp['pc'] = comp[ct]
		tmp.sort_values(['chrom','start'],ascending=[True,True],inplace=True)
		# f = pyBigWig.open(os.path.join(compartment_score_dir,f"{ct_name}.bw"), "w")
		# f.addHeader(chrom_sizes_list)
		for c in chrom_sizes.index: #order of chrom should be the same as chrom_sizes_list
			tmp1=tmp.loc[tmp['chrom'] == c]
			if tmp1.shape[0]==0:
				continue
			# f.addEntries(c, starts=(tmp1.start-1).tolist(),
			# 			 values=tmp1.pc.tolist(),span=res,validate=True)
			tmp1.to_csv(os.path.join(ct_dir, f"{c}.pc.bedGraph"),
											  sep='\t', header=False, index=False)
		# f.close()
	tmp = pd.DataFrame(index=[ct.replace('-', '_AND_').replace(' ','_').replace('+','_PLUS_') for ct in common_cts])
	tmp['matrix_path'] = '.'
	tmp['bed_path'] = '.'
	tmp['sample'] = tmp.index + '_100Kb'
	tmp['group'] = tmp.index
	input_path = os.path.join(group_dir, 'input.txt')
	tmp.to_csv(input_path, sep='\t', header=False, index=False)
	cmd = f"cd {group_dir} && {Rscript} ~/Software/dcHiC/dchicf.r --file {input_path} --pcatype analyze --dirovwt T --diffdir {group_name} --genome {genome}"
	# cmd = f"conda activate {dchic_env} && cd {group_dir} && Rscript ~/Software/dcHiC/dchicf.r --file {input_path} --pcatype analyze --dirovwt T --diffdir {group_name} --genome {genome} && conda deactivate"
	cmds.append(cmd)

shell_path=os.path.join(outdir,f"run_{group_col}_diff_comp.sh")
with open(shell_path, 'w') as f:
	for cmd in cmds:
		f.write(cmd + '\n')
cmd = f"source {shell_path}"
print(cmd)
if not prepare_only:
	os.system(cmd)